# Chuẩn bị dữ liệu huấn luyện

Chuẩn bị văn bản và trọng số lớp từ những bài đủ điều kiện. Giữ nguyên ID và thứ tự của ba tập đã lưu, không chia lại khi cập nhật nhãn. Bộ nhãn vẫn có 7 lớp.

Phiên bản `company_awards_listing_v2` bổ sung 6 quyết định nhãn và 3 sửa nội dung trong validation. Hai bài từng chờ đã được xác nhận giữ `financial_results`, với lý do trong `validation_revision.json`; nhãn và trạng thái gốc không đổi, validation vẫn có 159 bài. Train và test không thay đổi.

Notebook chưa tách từ, tokenize hoặc huấn luyện mô hình.

## 1. Thiết lập

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

In [ ]:
CURRENT_DIR = Path.cwd().resolve()
PROJECT_CANDIDATES = [CURRENT_DIR / 'news_processing', CURRENT_DIR, *CURRENT_DIR.parents]
PROJECT_DIR = next(
    (path for path in PROJECT_CANDIDATES
     if (path / 'src').is_dir() and (path / 'data').is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Không tìm thấy thư mục news_processing.')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

In [ ]:
from src.annotation_utils import apply_approved_label_corrections
from src.model_data_utils import (
    MODEL_SPLIT_COLUMNS,
    build_split_distribution_report,
    compute_balanced_class_weights,
    prepare_eligible_model_dataset,
    reuse_split_membership,
    stratified_train_val_test_split,
    verify_model_splits,
)

In [ ]:
ANNOTATION_PATH = PROJECT_DIR / 'data/annotation/news_type/news_manually_labeled.csv'
LABEL_CORRECTIONS_PATH = ANNOTATION_PATH.with_name('label_corrections.csv')
CLEAN_PATH = PROJECT_DIR / 'data/processed/news_clean.csv'
EXCLUDED_PATH = PROJECT_DIR / 'data/interim/news_excluded_records.csv'
MERGE_LOG_PATH = PROJECT_DIR / 'data/interim/duplicate_merge_log.csv'
LABEL_MAPPING_PATH = PROJECT_DIR / 'models/news_type_phobert_cnn/label_mapping.json'

SPLIT_DIR = PROJECT_DIR / 'data/splits/news_type'
REPORT_DIR = PROJECT_DIR / 'reports/news_type'
CLASS_WEIGHTS_PATH = PROJECT_DIR / 'models/news_type_phobert_cnn/class_weights.json'

TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42

## 2. Lấy những bài đủ điều kiện

Chỉ dùng bài đã chốt nhãn và đạt kiểm tra nội dung. Danh sách sửa nhãn đã được phê duyệt được đối chiếu theo ID và dấu kiểm nội dung; không suy nhãn bằng từ khóa. Ngoài danh sách này, nhãn thủ công được giữ nguyên.

In [ ]:
annotations = pd.read_csv(ANNOTATION_PATH, dtype=str, keep_default_na=False)
clean_news = pd.read_csv(CLEAN_PATH, dtype=str, keep_default_na=False)
excluded_records = pd.read_csv(EXCLUDED_PATH, dtype=str, keep_default_na=False)
duplicate_merge_log = pd.read_csv(MERGE_LOG_PATH, dtype=str, keep_default_na=False)
label_mapping = json.loads(LABEL_MAPPING_PATH.read_text(encoding='utf-8-sig'))
label2id = label_mapping['label2id']
display_names = label_mapping.get('display_names', {})

In [ ]:
if LABEL_CORRECTIONS_PATH.is_file():
    approved_corrections = pd.read_csv(
        LABEL_CORRECTIONS_PATH, dtype=str, keep_default_na=False,
    )
    annotations = apply_approved_label_corrections(annotations, approved_corrections)
print('Phiên bản nhãn:', label_mapping.get('label_policy_version', 'original'))

In [ ]:
revision_path = ANNOTATION_PATH.with_name('validation_revision.json')
if revision_path.is_file():
    revision_audit = json.loads(revision_path.read_text(encoding='utf-8-sig'))
    print('Bài còn chờ quyết định:', len(revision_audit['pending_reviews']))
    print('Bài đã xác nhận giữ nguyên nhãn:', len(revision_audit.get('resolved_reviews', [])))

In [ ]:
eligible_news, annotation_audit = prepare_eligible_model_dataset(
    annotations=annotations,
    clean_news=clean_news,
    excluded_records=excluded_records,
    duplicate_merge_log=duplicate_merge_log,
    label2id=label2id,
)

print('Bài trong file annotation:', len(annotations))
print('Bài đủ điều kiện:', len(eligible_news))
print('Bài chưa dùng để huấn luyện:', len(annotations) - len(eligible_news))
print('Số nhãn huấn luyện:', eligible_news['label_id'].nunique())

## 3. Văn bản đầu vào

Ghép tiêu đề, tóm tắt bổ sung và thân bài. Quy tắc bỏ tóm tắt dựa trên so sánh sau chuẩn hóa và khớp tiền tố; không chỉ kiểm tra trùng nguyên văn. Tóm tắt trùng tiêu đề hoặc không có nội dung cũng không được ghép thêm.

In [ ]:
character_lengths = eligible_news['input_text'].str.len()
whitespace_unit_counts = eligible_news['input_text'].str.split().str.len()
skipped_summary_count = int(eligible_news['summary_redundant'].sum())

input_text_summary = pd.DataFrame([
    {'metric': 'Bài đủ điều kiện', 'value': len(eligible_news)},
    {'metric': 'Bài không ghép thêm summary', 'value': skipped_summary_count},
    {'metric': 'Bài ghép thêm summary', 'value': len(eligible_news) - skipped_summary_count},
    {'metric': 'Số ký tự ngắn nhất', 'value': int(character_lengths.min())},
    {'metric': 'Số ký tự trung vị', 'value': float(character_lengths.median())},
    {'metric': 'Số đơn vị cách bởi khoảng trắng, trung vị',
     'value': float(whitespace_unit_counts.median())},
])
display(input_text_summary)

Bảng này mô tả độ dài trước tokenizer, chưa cho biết bài nào sẽ bị cắt khi đưa vào mô hình.

## 4. Chia train, validation và test

Lần đầu chia phân tầng theo nhãn với seed cố định. Khi đã có đủ ba tập, dùng lại đúng ID và thứ tự đã lưu; sửa nhãn không được làm bài chuyển tập. Các nhãn thủ công được chốt sau khi baseline đã khóa chỉ là nhãn tham chiếu cho đầu ra, không tự động đi vào cohort train/validation/test hiện hành. Ba split baseline tiếp tục giữ đúng ID, thứ tự và quy mô 740/159/159.

In [ ]:
split_paths = {name: SPLIT_DIR / f'{name}.csv' for name in ('train', 'validation', 'test')}
existing_split_count = sum(path.is_file() for path in split_paths.values())
if existing_split_count not in (0, 3):
    raise FileNotFoundError('Thiếu một phần bộ split; không tự chia lại dữ liệu.')
existing_splits = {
    name: pd.read_csv(path, dtype=str, keep_default_na=False)
    for name, path in split_paths.items() if path.is_file()
}

In [ ]:
if existing_splits:
    saved_model_ids = {
        news_id for frame in existing_splits.values()
        for news_id in frame['news_id'].tolist()
    }
    in_baseline = eligible_news['news_id'].isin(saved_model_ids)
    model_dataset = eligible_news.loc[in_baseline, MODEL_SPLIT_COLUMNS].copy()
    manual_reference_not_in_model = eligible_news.loc[~in_baseline].copy()
    train_data, validation_data, test_data = reuse_split_membership(
        model_dataset, existing_splits
    )
    print('Nhãn manual ngoài cohort baseline (không dùng để train/evaluate):',
          len(manual_reference_not_in_model))
else:
    model_dataset = eligible_news[MODEL_SPLIT_COLUMNS].copy()
    manual_reference_not_in_model = eligible_news.iloc[:0].copy()
    train_data, validation_data, test_data = stratified_train_val_test_split(
        model_dataset, train_ratio=TRAIN_RATIO, val_ratio=VALIDATION_RATIO,
        test_ratio=TEST_RATIO, group_column='label_id', random_seed=RANDOM_SEED,
    )

In [ ]:
split_sizes = pd.Series({
    'train': len(train_data),
    'validation': len(validation_data),
    'test': len(test_data),
}, name='article_count')
display(split_sizes.to_frame())

In [ ]:
split_distribution = build_split_distribution_report(
    train_data,
    validation_data,
    test_data,
    label2id=label2id,
    display_names=display_names,
)
display(split_distribution)

## 5. Trọng số lớp

Tính trọng số nghịch đảo tần suất chỉ từ train: số bài train chia cho tích số lớp và số bài train của lớp đó. Trọng số là cấu hình cho hàm loss, chưa phải bằng chứng mô hình dự đoán tốt hơn.

In [ ]:
class_weights_table = compute_balanced_class_weights(train_data, label2id=label2id)
display(class_weights_table)

## 6. Kiểm tra trùng giữa các tập

Đối chiếu số mẫu, trường bắt buộc, nhãn và các khóa nhận diện bài. Kiểm tra trùng dựa trên ID, URL, tiêu đề chuẩn hóa và nội dung chính xác.

In [ ]:
split_checks = verify_model_splits(
    full_eligible=model_dataset,
    train_df=train_data,
    val_df=validation_data,
    test_df=test_data,
    label2id=label2id,
)
split_validation_status = 'PASS' if split_checks['issue_count'].eq(0).all() else 'FAIL'

display(split_checks)
print('Kết quả kiểm tra:', split_validation_status)
assert split_validation_status == 'PASS', 'Dừng lưu: kiểm tra chia tập chưa đạt.'

Không có trùng lặp theo các khóa đã kiểm tra. Những bài khác nhau cùng nói về một sự kiện chưa được gom nhóm; kết quả này không chứng minh các tập độc lập theo sự kiện hoặc thời gian.

## 7. Chuẩn bị file đầu ra

Bảng dưới liệt kê các file sẽ lưu. Chỉ chạy các cell ghi file sau khi kiểm tra chia tập đạt yêu cầu.

In [ ]:
export_tables = {
    SPLIT_DIR / 'train.csv': train_data,
    SPLIT_DIR / 'validation.csv': validation_data,
    SPLIT_DIR / 'test.csv': test_data,
    REPORT_DIR / 'split_distribution.csv': split_distribution,
}

export_summary = pd.DataFrame([
    {'file': path.relative_to(PROJECT_DIR).as_posix(), 'rows': len(table)}
    for path, table in export_tables.items()
] + [{
    'file': CLASS_WEIGHTS_PATH.relative_to(PROJECT_DIR).as_posix(),
    'rows': len(class_weights_table),
}])
display(export_summary)

In [ ]:
class_weights_payload = {
    'train_size': int(len(train_data)),
    'num_classes': int(len(class_weights_table)),
    'weights_by_id': {
        str(int(row.label_id)): float(row.class_weight)
        for row in class_weights_table.itertuples(index=False)
    },
    'weights_by_label': {
        str(row.news_type): float(row.class_weight)
        for row in class_weights_table.itertuples(index=False)
    },
}

In [ ]:
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

for output_path, output_table in export_tables.items():
    output_table.to_csv(output_path, index=False, encoding='utf-8-sig')

In [ ]:
CLASS_WEIGHTS_PATH.parent.mkdir(parents=True, exist_ok=True)
CLASS_WEIGHTS_PATH.write_text(
    json.dumps(class_weights_payload, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)

## Bước tiếp theo

Lưu các file sau khi bảng kiểm tra đạt yêu cầu. Dùng đúng phiên bản nhãn cho lần huấn luyện tiếp theo; Accuracy/F1 của checkpoint cũ không phải kết quả trên bộ nhãn đã sửa. Notebook này không huấn luyện hoặc đánh giá model.